---
![LOGO UTN FRCU](https://frcu.utn.edu.ar/images/recursos/logos/MARCA_FACULTAD_REGIONAL_NEGRO.png)
# BiblioIA: Sistema de Gestión de Biblioteca con Agente de Inteligencia Artificial"
- Materia: Base de Datos
- Año / Carrera: 3° año — Ing. en Sistemas de Información
- Docentes: Ing. Francisco Fazzio, Ing. Pablo Colombo
- Ciclo lectivo: 2026
- Grupo : Grupo Pandas
- Integrantes : Cheveste Baloni Ulises, Delfino Jeremias, Gallegos Marco, Ocampo Emmanuel


---
## 1. Configuración
carga .env, conecta a MySQL.

### 1.1 Instalación de librerías

In [16]:
pip install mysql-connector-python pandas google-genai python-dotenv


OS   ›  Arch Linux x86_64
KER  ›  Linux 7.0.12-arch1-1
PKG  ›  1699 (pacman)
WM   ›  Hyprland 0.55.4 (Wayland)

Note: you may need to restart the kernel to use updated packages.


### 1.2 Configuración

In [17]:
import os
from dotenv import load_dotenv
import mysql.connector
import pandas as pd
import ipywidgets as widgets
from google import genai

load_dotenv()
DB_CONF = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT",  "3306"),
    "database": os.getenv("DB_NAME", "biblioia"),
    "user": os.getenv("DB_USER", "root"),
    "password": os.getenv("DB_PASSWORD", "")
}

LLM_API_KEY= os.getenv("LLM_API_KEY", "")




---
## 2. Inspección del esquema
Carga structure de tablas para el prompt

In [18]:
sys_prompt = """ Sos un Administrador de bases de datos de una biblioteca. Responde las consultas en lenguaje natural únicamente devolviendo una consulta en MySQL
    sabiendo que el schema de la base de datos en MySQL de esta biblioteca es el siguiente:
    1. **Genero** (Categorías de libros)
    - id_genero (PK, auto_increment)
    - nombre (VARCHAR 100, UNIQUE, NOT NULL)
    - descripcion (TEXT)

    2. **Autor** (Autores de libros)
    - id_autor (PK, auto_increment)
    - nombre (VARCHAR 100, NOT NULL)
    - apellido (VARCHAR 100, NOT NULL)
    - nacionalidad (VARCHAR 100)

    3. **Socio** (Miembros de la biblioteca)
    - id_socio (PK, auto_increment)
    - dni (VARCHAR 20, UNIQUE, NOT NULL)
    - nombre (VARCHAR 100, NOT NULL)
    - apellido (VARCHAR 100, NOT NULL)
    - email (VARCHAR 150, UNIQUE, NOT NULL)
    - fecha_alta (DATE, NOT NULL)
    - estado (VARCHAR 20, DEFAULT 'activo') - Valores: 'activo', 'suspendido', 'baja'

    4. **Libro** (Catálogo de libros)
    - isbn (VARCHAR 20, PK)
    - titulo (VARCHAR 255, NOT NULL)
    - anio_publicacion (INT)
    - stock_total (INT, DEFAULT 0)
    - stock_disponible (INT, DEFAULT 0)
    - id_genero (FK → Genero)

    5. **Autor_libro** (Relación muchos-a-muchos entre Autores y Libros)
    - id_autor (FK → Autor)
    - isbn (FK → Libro)
    - PK: (id_autor, isbn)

    6. **Ejemplar** (Copias físicas de libros)
    - id_ejemplar (PK, auto_increment)
    - isbn (FK → Libro, NOT NULL)
    - nro_ejemplar (INT, NOT NULL)
    - estado_fisico (VARCHAR 20, DEFAULT 'bueno') - Valores: 'bueno', 'regular', 'reparacion', 'baja'

    7. **Prestamo** (Registro de préstamos)
    - id_prestamo (PK, auto_increment)
    - id_socio (FK → Socio, NOT NULL)
    - id_ejemplar (FK → Ejemplar, NOT NULL)
    - fecha_prestamo (DATE, NOT NULL)
    - fecha_vencimiento (DATE, NOT NULL)
    - fecha_devolucion (DATE, NULL)
    - estado (VARCHAR 20, DEFAULT 'activo') - Valores: 'activo', 'devuelto', 'vencido'

    8. **Sancion** (Sanciones a socios)
    - id_sancion (PK, auto_increment)
    - id_socio (FK → Socio, NOT NULL)
    - tipo (VARCHAR 50, NOT NULL)
    - fecha_inicio (DATE, NOT NULL)
    - fecha_fin (DATE, NOT NULL)
    - motivo (TEXT)

    INSTRUCCIONES IMPORTANTES:
    - Cuando generes consultas SQL, asegúrate de que sean válidas para MySQL.
    - Usa INNER JOIN para relaciones requeridas y LEFT JOIN cuando sea necesario.
    - Siempre valida que los tipos de datos sean correctos.
    - Tus respuestas deben contener únicamente la consulta SQL, sin ningún tipo de texto adicional, pues serán cargadas a la base de datos. 
    """

---
## 3. Función text_to_sql
Llama a la API y retorna SQL.

In [19]:
# Cargar prompt y toda la cuestión lol
def conectar_DB(query):
    try:
        db= mysql.connector.connect(**DB_CONF)
        resultado = pd.read_sql(query,db)
        print(resultado.head())
        resultado.to_csv('Test.csv')
        db.close() #close the connection
    except Exception as e:
        db.close()
        print(str(e))

def consultar(consulta):
    
    client = genai.Client(api_key=LLM_API_KEY)

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=f"{sys_prompt}\n\nConsulta: {consulta_natural}"
    )
    client.close()
    conectar_DB(response.text)



---
## 4. Función agente_responder(pregunta)
Orquesta todo y muestra resultado.

In [20]:
consulta_natural = input('Ingrese su consulta: ')
consultar(consulta_natural)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

---
## 5. Demo interactivo
Widget ipywidgets o input() para preguntas libres

---
## 6. Módulo de recomendaciones: función recomendar_para(id_socio)